# CDP simulation — GPU (CuPy) version

Same physics, solver and sweeps as `CDP_Simulation_v8.1_real_setup.ipynb`, adapted to run on a CUDA GPU such as Colab's T4:

- all large arrays (fields, FFTs, masks, the sparse detector matrix) live on the GPU through CuPy (`xp` in the code);
- random numbers (masks, noise, start vectors) are still drawn on the CPU, so results are identical for the same seed;
- results are brought back to NumPy (`to_cpu`) for metrics and plots;
- the sweeps run one simulation at a time (the GPU does the parallel work).

**On Colab:** Runtime → Change runtime type → *T4 GPU*, then Runtime → Run all. Without a GPU the notebook falls back to NumPy on the CPU.

In [ ]:
# GPU version for Google Colab: Runtime > Change runtime type > T4 GPU, then run all cells.
# CuPy is preinstalled on Colab's GPU runtimes. Without a GPU the notebook runs on the CPU with NumPy.
!nvidia-smi -L || echo "no NVIDIA GPU visible: the notebook will run on the CPU"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import math
import multiprocessing
from scipy.ndimage import map_coordinates
import scipy.sparse as sp

# [GPU] Run on a CUDA GPU with CuPy when one is available (Colab: Runtime > Change runtime
# type > T4 GPU; CuPy is preinstalled there). Without a GPU the same code runs on NumPy.
# `xp` is the array library used for all heavy arrays; to_cpu() brings results back to NumPy.
try:
    import cupy as cp
    import cupyx.scipy.sparse as cpsp
    GPU = cp.cuda.runtime.getDeviceCount() > 0
except Exception:
    GPU = False

if GPU:
    xp = cp
    print("Running on GPU:", cp.cuda.runtime.getDeviceProperties(0)["name"].decode())
else:
    xp = np
    print("No GPU found: running on the CPU with NumPy")

fft2, ifft2 = xp.fft.fft2, xp.fft.ifft2

def to_cpu(a):
    '''GPU array -> NumPy array (does nothing on the CPU)'''
    return cp.asnumpy(a) if GPU and isinstance(a, cp.ndarray) else a

def cubic_kernel(t, a=-0.5):
    '''Keys cubic interpolation kernel (support |t| < 2)'''

    t = np.abs(t)

    return np.where(t <= 1, (a + 2) * t**3 - (a + 3) * t**2 + 1,
           np.where(t < 2, a * t**3 - 5 * a * t**2 + 8 * a * t - 4 * a, 0.0))

class SamplingOperator:

    def __init__(self, n_prop, dx_prop, n_det, dx_det, shift_x=0.0, shift_y=0.0, rotation_deg=0.0, origin=None, tilt_x_deg=0.0, tilt_y_deg=0.0, subsamples=1, precision="double"):
        """
        Maps propagated field on propagation grid
        to detector field on detector grid.

        n_prop : propagation grid size (use Propagator.n_out)
        dx_prop : propagation grid pitch (use Propagator.dx_out)

        n_det : detector grid size
        dx_det : detector pixel pitch

        shift_x, shift_y : physical displacement of the detector centre
                           from the optical axis [m]
        rotation_deg : rotation of the detector about its centre,
                       counter-clockwise from x towards y [deg]
        origin : propagation-grid index of the optical axis
                 (default n_prop/2, use Propagator.origin_out)
        tilt_x_deg, tilt_y_deg : out-of-plane tilt of the detector about its
                                 centre, about the x then the y axis [deg]
        subsamples : each pixel is sampled at subsamples x subsamples points
                     spread over its area; the pixel reads the mean intensity
        precision : "double" or "single" (single is ~1.5x faster, ample accuracy)
        """

        self.n_prop = n_prop
        self.n_det = n_det
        self.n_sub = subsamples ** 2

        # [SPEED] single precision halves memory traffic in the sparse products
        self.rdtype = np.float32 if precision == "single" else np.float64
        self.cdtype = np.complex64 if precision == "single" else np.complex128

        # [FIX 1] the optical axis is not necessarily at n_prop/2 once the
        # SLM is embedded in a larger (zero-padded) propagation grid
        if origin is None:
            origin = n_prop / 2

        theta = np.deg2rad(rotation_deg)

        # detector coordinates (physical)
        x_det = (np.arange(n_det) - n_det/2) * dx_det
        y_det = (np.arange(n_det) - n_det/2) * dx_det

        Xd, Yd = np.meshgrid(x_det, y_det, indexing="ij")

        # [DETECTOR] sample points at the centres of subsamples x subsamples
        # equal sub-cells of each pixel (in the detector's own coordinates,
        # so they rotate and tilt with the pixel)
        o = ((np.arange(subsamples) + 0.5) / subsamples - 0.5) * dx_det
        offsets = [(ox, oy) for ox in o for oy in o]

        # [FIX 3] forward transform: detector pixel at local (Xd, Yd)
        # sits at R(theta) (Xd, Yd) + shift in the propagation plane
        ct = np.cos(theta)
        st = np.sin(theta)

        # out-of-plane tilt: rotate the detector plane about x, then y,
        # and project onto the propagation plane (the axial offset of each
        # pixel is neglected: < 1 % effect for tilts up to 10 deg here)
        tx, ty = np.deg2rad(tilt_x_deg), np.deg2rad(tilt_y_deg)
        cx, sx = np.cos(tx), np.sin(tx)
        cy, sy = np.cos(ty), np.sin(ty)

        # power on a tilted pixel scales with cos(alpha) = cx*cy,
        # so the amplitude scales with its square root
        obliquity = np.sqrt(cx * cy)

        n_pix = n_det * n_det
        rows, cols, vals = [], [], []
        invalid = np.zeros((n_det, n_det), dtype=bool)

        for k, (ox, oy) in enumerate(offsets):

            # 1) in-plane rotation, 2) tilt, 3) shift in lab coordinates
            X_rot = ct*(Xd + ox) - st*(Yd + oy)
            Y_rot = st*(Xd + ox) + ct*(Yd + oy)

            Xp = X_rot*cy + Y_rot*sx*sy + shift_x
            Yp = Y_rot*cx + shift_y

            # convert physical coordinates
            # to propagation-grid indices

            Ix = Xp/dx_prop + origin
            Iy = Yp/dx_prop + origin

            x0 = np.floor(Ix).astype(np.int64)
            y0 = np.floor(Iy).astype(np.int64)

            # [DETECTOR] cubic interpolation uses the 4 x 4 nearest grid points
            valid = (x0 - 1 >= 0) & (x0 + 2 < n_prop) & (y0 - 1 >= 0) & (y0 + 2 < n_prop)
            invalid |= ~valid

            row = k * n_pix + np.arange(n_pix)

            for i in range(-1, 3):
                wx = cubic_kernel(Ix - (x0 + i))
                xi = np.clip(x0 + i, 0, n_prop - 1)

                for j in range(-1, 3):
                    wy = cubic_kernel(Iy - (y0 + j))
                    yj = np.clip(y0 + j, 0, n_prop - 1)

                    rows.append(row)
                    cols.append((xi * n_prop + yj).ravel())
                    vals.append((wx * wy * valid * obliquity).ravel())

        # [FIX 1] pixels outside the grid read 0, which a real camera would not
        n_outside = np.count_nonzero(invalid)
        if n_outside > 0:
            print(f"WARNING: {n_outside} of {n_det**2} detector pixels lie outside "
                  f"the propagation grid and will read 0. Increase n_prop.")

        # [DETECTOR] the whole mapping is one sparse matrix, which is much
        # faster than np.add.at for the adjoint
        self.M = sp.csr_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))),
                               shape=(self.n_sub * n_pix, n_prop * n_prop)).astype(self.rdtype)
        self.MT = self.M.T.tocsr()

        if GPU:
            # [GPU] move the matrices to the GPU; cuSPARSE wants the same type as the
            # field it multiplies, so the (real) weights are stored as complex numbers
            self.M = cpsp.csr_matrix(self.M.astype(self.cdtype))
            self.MT = cpsp.csr_matrix(self.MT.astype(self.cdtype))

    def forward(self, U):
        """
        U shape:
            (L, n_prop, n_prop)

        returns the field at every sample point:
            (L, n_sub, n_det, n_det)
        """

        L = U.shape[0]
        U = U.astype(self.cdtype, copy=False)

        return (self.M @ xp.ascontiguousarray(U.reshape(L, -1).T)).T.reshape(L, self.n_sub, self.n_det, self.n_det)

    def adjoint(self, V):
        """
        V shape:
            (L, n_sub, n_det, n_det)

        returns:
            (L, n_prop, n_prop)
        """

        L = V.shape[0]
        V = V.astype(self.cdtype, copy=False)

        return (self.MT @ xp.ascontiguousarray(V.reshape(L, -1).T)).T.reshape(L, self.n_prop, self.n_prop)

    def intensity(self, V):
        """
        [DETECTOR] pixel value = mean intensity over the pixel's sample points
        V shape: (L, n_sub, n_det, n_det) -> (L, n_det, n_det)
        """

        return xp.mean(xp.abs(V) ** 2, axis=1)

class Propagator:

    def __init__(self, n_in, n, dx, wavelength, z, fill_factor=1.0, upsample=1, precision="double"):
        """
        n_in : SLM grid size (the field is defined on n_in x n_in)
        n : propagation grid size, n >= n_in (the SLM is zero-padded to n x n)
        fill_factor : fraction of each SLM pixel's area that is active
                      (square aperture of side dx * sqrt(fill_factor))
        upsample : the propagated field is returned on a grid upsample times
                   finer (pitch dx / upsample) by exact Fourier interpolation
        precision : "double" or "single" (single is ~1.5x faster, ample accuracy)
        """

        # [FIX 1] the SLM field is embedded in a larger zero-padded grid so
        # light leaving the SLM aperture does not wrap around onto the camera
        self.n_in = n_in
        self.n = n
        self.dx = dx
        self.wavelength = wavelength
        self.z = z
        self.fill_factor = fill_factor

        # [SPEED] complex type of all propagation arrays
        self.cdtype = np.complex64 if precision == "single" else np.complex128

        self.offset = (n - n_in) // 2
        # grid index of the optical axis (SLM pixel i is at (i - n_in/2) * dx)
        self.origin = self.offset + n_in / 2

        # [DETECTOR] output grid: upsample times finer than the SLM grid.
        # Zero-padding the spectrum gives the exact band-limited field
        # between the SLM-pitch samples (instead of interpolating it)
        self.m = upsample
        self.n_out = n * upsample
        self.dx_out = dx / upsample
        self.origin_out = self.origin * upsample
        # [SPEED] number of non-negative / negative frequencies, used to copy the
        # spectrum between the two grids with plain slices (faster than fancy indexing)
        self.h = (n + 1) // 2
        self.k = n // 2

        fx = np.fft.fftfreq(n, d=dx)
        fy = np.fft.fftfreq(n, d=dx)
        FX, FY = np.meshgrid(fx, fy)

        term = (1 - (wavelength * FX) ** 2 - (wavelength * FY) ** 2)
        k = 2 * np.pi / wavelength

        # [FIX 1] band-limited angular spectrum (Matsushima & Shimobaba 2009):
        # drop frequencies where the transfer-function phase is undersampled
        df = 1 / (n * dx)
        f_limit = 1 / (wavelength * np.sqrt((2 * df * z) ** 2 + 1))
        band = (np.abs(FX) <= f_limit) & (np.abs(FY) <= f_limit)

        # SLM pixel shape: each pixel is a square aperture of side a, whose
        # spectrum is sinc(a fx) sinc(a fy). The amplitude of the main
        # diffraction order scales with the aperture area (fill_factor);
        # the rest of the transmitted light goes into higher orders, which
        # a grid of pitch dx cannot represent
        a = dx * np.sqrt(fill_factor)
        pixel_envelope = np.sinc(a * FX) * np.sinc(a * FY)

        self.H = xp.asarray((np.exp(1j * k * z * np.sqrt(term.astype(complex))) * band * fill_factor * pixel_envelope).astype(self.cdtype))
        self.H_adj = xp.conj(self.H)

    def pad(self, U):

        out = xp.zeros(U.shape[:-2] + (self.n, self.n), dtype=self.cdtype)
        out[..., self.offset:self.offset + self.n_in, self.offset:self.offset + self.n_in] = U

        return out

    def crop(self, V):

        return V[..., self.offset:self.offset + self.n_in, self.offset:self.offset + self.n_in]

    def forward(self, U):
        '''(L, n_in, n_in) on the SLM -> (L, n_out, n_out) on the output grid'''

        A = fft2(self.pad(U), axes=(-2,-1), norm="ortho") * self.H[None,:,:]

        if self.m == 1:
            return ifft2(A, axes=(-2,-1), norm="ortho")

        # [DETECTOR] zero-pad the spectrum to n_out x n_out
        B = xp.zeros(A.shape[:-2] + (self.n_out, self.n_out), dtype=self.cdtype)
        n, N, h, k = self.n, self.n_out, self.h, self.k
        B[..., :h, :h] = A[..., :h, :h]
        B[..., :h, N-k:] = A[..., :h, n-k:]
        B[..., N-k:, :h] = A[..., n-k:, :h]
        B[..., N-k:, N-k:] = A[..., n-k:, n-k:]

        return ifft2(B, axes=(-2,-1), norm="ortho") * self.m

    def adjoint(self, V):
        '''(L, n_out, n_out) on the output grid -> (L, n_in, n_in) on the SLM'''

        A = fft2(V.astype(self.cdtype, copy=False), axes=(-2,-1), norm="ortho")

        if self.m > 1:
            B = A
            A = xp.empty(B.shape[:-2] + (self.n, self.n), dtype=self.cdtype)
            n, N, h, k = self.n, self.n_out, self.h, self.k
            A[..., :h, :h] = B[..., :h, :h]
            A[..., :h, n-k:] = B[..., :h, N-k:]
            A[..., n-k:, :h] = B[..., N-k:, :h]
            A[..., n-k:, n-k:] = B[..., N-k:, N-k:]
            A *= self.m

        return self.crop(ifft2(A * self.H_adj[None,:,:], axes=(-2,-1), norm="ortho"))

def choose_prop_grid(n_slm, dx_slm, n_det, dx_det, wavelength, z, shift_max=0.0, rotation_max_deg=0.0, margin=2.0):
    '''[FIX 1] Smallest power-of-two propagation grid that
       - contains the SLM and the (shifted, rotated) detector,
       - is wide enough that light diffracted off the SLM edge cannot
         wrap around (FFT periodicity) and land on the detector,
       - samples the angular-spectrum transfer function: n >= wavelength z / dx^2'''

    slm_half = n_slm * dx_slm / 2

    theta = np.deg2rad(rotation_max_deg)
    det_half = n_det * dx_det / 2 * (abs(np.cos(theta)) + abs(np.sin(theta))) + shift_max

    # largest transverse displacement of light diffracted by the SLM grid
    s = wavelength / (2 * dx_slm)
    spread = z * s / np.sqrt(1 - s ** 2)

    width = margin * max(slm_half + spread + det_half, 2 * det_half, 2 * slm_half)
    n_min = max(width / dx_slm, wavelength * z / dx_slm ** 2)

    return int(2 ** np.ceil(np.log2(n_min)))

def define_sample(n, sigma, dx_slm, aberrations=None, R=None):
    '''Gaussian beam with optional aberrations.
       aberrations: dict of coefficients in radians (missing = 0)
       R: radius the aberrations are normalised to (default: half the SLM width)'''

    a = aberrations or {}
    if R is None:
        R = n * dx_slm / 2

    x = (np.arange(n) - n/2) * dx_slm
    y = (np.arange(n) - n/2) * dx_slm
    X, Y = np.meshgrid(x, y, indexing="ij")

    amplitude = np.exp(-(X ** 2 + Y ** 2) / (2 * sigma ** 2))

    u, v = X / R, Y / R          # normalised coordinates, edge of SLM = 1
    r2 = u**2 + v**2

    phase = (a.get("tilt_x", 0)    * u
           + a.get("tilt_y", 0)    * v
           + a.get("defocus", 0)   * (2*r2 - 1)
           + a.get("astig_0", 0)   * (u**2 - v**2)
           + a.get("astig_45", 0)  * (2*u*v)
           + a.get("coma_x", 0)    * (3*r2 - 2) * u
           + a.get("coma_y", 0)    * (3*r2 - 2) * v
           + a.get("spherical", 0) * (6*r2**2 - 6*r2 + 1))

    return amplitude * np.exp(1j * phase)

def define_masks(L, n, rng):
    '''Define the masks'''

    # [FIX 5] random numbers come from a seeded generator
    i = rng.integers(0, 256, size=(L, n, n))
    phi = 2*np.pi * i / 256

    return xp.asarray(np.exp(1j * phi))      # [GPU] random numbers on the CPU (reproducible), masks on the GPU

def effective_masks(D, eta, psi=0.0):
    '''[ZERO ORDER] masks including the light the SLM leaves unmodulated.
       eta : fraction of the power that is not modulated (0 = ideal SLM)
       psi : phase of the unmodulated light relative to the modulated light [rad]

       Each pixel multiplies the beam by sqrt(1-eta) e^{-i phi} + sqrt(eta) e^{i psi}
       (the code applies conj(D)), so the zeroth order is just part of the mask
       and every other function works unchanged. On average over random masks
       the total power is conserved.'''

    return np.sqrt(1 - eta) * D + np.sqrt(eta) * np.exp(-1j * psi)

def y_i(D, X, propagator, sampler):
    '''Simulates a measurement'''

    inner = sampler.forward(forward(X, np.conj(D), propagator))

    # [DETECTOR] mean intensity over each pixel's area
    return sampler.intensity(inner)

def forward(X, D_conj, propagator):
    '''Propagates the wave X forward from the mask plane to the detector plane to form Y'''
    '''Y_l = A(X) = P(D_l^* X)'''
    '''D_l is the mask, P is forward propagation'''
    '''X has shape (n, n)'''

    U = D_conj * xp.asarray(X)[None, :, :]
    return propagator.forward(U)


def adjoint(Y, D, propagator):
    '''Propagates the wave Y backward from the detector plane to the mask plane to form X'''
    '''X = A^*(Y) = Σ_l D_l P^*(Y_l)'''
    '''D_l is the mask, P^* is back propagation'''
    '''Y has shape (L, n, n)'''

    B = propagator.adjoint(Y)
    return xp.sum(D * B, axis=0)


def add_camera_noise(y, rng, full_well=20000, peak_fill=0.8, read_noise=3.0,
                     dark_e=0.0, bit_depth=12, offset_adu=100):
    '''Realistic camera noise. Returns noisy data in the same units as y.
       full_well  : max electrons per pixel            [e-]
       peak_fill  : brightest pixel as a fraction of full well (sets exposure)
       read_noise : Gaussian read noise                 [e- rms]
       dark_e     : dark electrons per pixel per frame  [e-]
       bit_depth  : ADC bits
       offset_adu : black level                         [ADU]'''

    y = to_cpu(y)                                        # [GPU] noise is drawn on the CPU

    # exposure: electrons per unit of y, the same for all frames
    scale = peak_fill * full_well / y.max()

    # shot noise (signal + dark current), then read noise
    e = rng.poisson(y * scale + dark_e).astype(float)
    e += rng.normal(0.0, read_noise, y.shape)

    # saturation, then digitisation with offset
    e = np.clip(e, 0, full_well)
    gain = full_well / (2**bit_depth - 1)               # e- per ADU
    adu = np.clip(np.round(e / gain) + offset_adu, 0, 2**bit_depth - 1)

    # calibration: subtract offset and dark level, back to the units of y
    # (negative values are kept so the noise stays unbiased)
    return xp.asarray(((adu - offset_adu) * gain - dark_e) / scale)

def spectral_init(y, D, propagator, sampler, iter, tol, rng):
    '''Computes the dominant eigenvector of A_l^* diag(y_l) A_l'''

    n = D.shape[1]

    z = xp.asarray(rng.standard_normal((n, n)) + 1j * rng.standard_normal((n, n)))
    z /= xp.linalg.norm(z)

    D_conj = np.conj(D)

    for _ in range(iter):
        Az = sampler.forward(forward(z, D_conj, propagator))
        weighted = sampler.adjoint(y[:, None] * Az)
        w = adjoint(weighted, D, propagator)

        z_new = w / xp.linalg.norm(w)
        corr = xp.abs(xp.vdot(z_new, z))

        if 1.0 - corr < tol:
            z = z_new
            break

        z = z_new

    # [INIT] Candes et al.: give z the size implied by the data. Scale it so the
    # predicted camera images carry the same total intensity as the measured ones
    # (the exact version of their lambda^2 = n sum(y) / sum(|a_r|^2) for this operator)
    Az = sampler.intensity(sampler.forward(forward(z, D_conj, propagator)))
    z *= xp.sqrt(xp.sum(y) / xp.sum(Az))

    return z

def Wirtinger_iteration(x0, y, D, propagator, sampler, iter, tol, tau0, mu_max, accelerate=True, return_iters=False, normalize_step=True):
    '''Solves the minimzation f(X) = Σ_l(|A_l(X)|^2 - y_l)^2 using Wirtinger Flow
       accelerate : Nesterov momentum with adaptive restart (same result, 2-4x fewer iterations)
       return_iters : also return the number of iterations used
       normalize_step : divide the step by the curvature scale of the problem, so mu_max
                        is dimensionless and stays valid when the beam brightness, the
                        optics or the noise change (Candes et al. use mu / |z0|^2)'''

    x = x0.copy()
    x_prev = x.copy()
    L = D.shape[0]
    D_conj = np.conj(D)
    f_prev = np.inf

    # [INIT] curvature scale ~ mean(y) * sum(y) / (L |x0|^2); its inverse multiplies the step
    step_scale = (L**2 * y[0].size * xp.real(xp.vdot(x0, x0))) / xp.sum(y)**2 if normalize_step else 1.0

    v_prev = g_prev = None
    mu_prev = 0.0

    for tau in range(iter):
        mu = min(1 - np.exp(-tau/tau0), mu_max)

        # [SPEED] Nesterov: take the gradient step from a look-ahead point
        v = x + (tau / (tau + 3)) * (x - x_prev) if accelerate else x

        inner = sampler.forward(forward(v, D_conj, propagator))

        # [DETECTOR] the model is the mean of |field|^2 over each pixel's
        # sample points, so the gradient averages over them as well
        residual = sampler.intensity(inner) - y
        f = xp.sum(residual ** 2)

        if accelerate and f > f_prev and v is not x:
            # [SPEED] restart the momentum whenever the error goes up
            x_prev = x.copy()
            v = x
            inner = sampler.forward(forward(v, D_conj, propagator))
            residual = sampler.intensity(inner) - y
            f = xp.sum(residual ** 2)

        if v_prev is not None and (not xp.isfinite(f) or f > f_prev * (1 + 1e-7)):
            # [SAFEGUARD] even a plain gradient step increased the error: the step was
            # too long. Redo the last step at half the length (this cannot diverge)
            step_scale *= 0.5
            x = v_prev - mu_prev * step_scale * g_prev
            x_prev = v_prev.copy()
            continue

        g = adjoint(sampler.adjoint(residual[:, None] * inner), D, propagator) / (L * sampler.n_sub)

        f_prev, v_prev, g_prev, mu_prev = f, v, g, mu
        x_prev = x
        x = v - mu * step_scale * g

        grad_norm = xp.sqrt(xp.real(xp.vdot(g, g)))
        x_norm = xp.sqrt(xp.real(xp.vdot(x, x)))

        if grad_norm / x_norm < tol:
            break

    return (x, tau + 1) if return_iters else x

def generate_measurement(x, propagator, sampler, noise, L, n, rng, zero_order_power=0.0, zero_order_phase=0.0):
    ''''Generate the masks D and the measurements y_i
        noise : dict of add_camera_noise parameters, or None for noise-free data
        zero_order_power, zero_order_phase : unmodulated SLM light in the data
        Returns the programmed masks D (what the experimenter knows)'''

    start = time.perf_counter()
    D = define_masks(L, n, rng)

    # [ZERO ORDER] the camera sees the effective masks, not the programmed ones
    y = y_i(effective_masks(D, zero_order_power, zero_order_phase), x, propagator, sampler)
    if noise is not None:
        y = add_camera_noise(y, rng, **noise)
    end = time.perf_counter()

    print(f"Ran in {end - start}s")

    return y, D

def reconstruct_x(y, D, propagator, sampler, iter_WF, iter_SI, tol_WF, tol_SI, tau0, mu_max, plot_x, sigma, dx_slm, rng):
    '''Reconstruct the object x given y, D and H'''

    start = time.perf_counter()

    start_spec = time.perf_counter()
    x0 = spectral_init(y, D, propagator, sampler, iter_SI, tol_SI, rng)
    end_spec = time.perf_counter()

    start_Wirt = time.perf_counter()
    x_r = Wirtinger_iteration(x0, y, D, propagator, sampler, iter_WF, tol_WF, tau0, mu_max)
    end_Wirt = time.perf_counter()

    end = time.perf_counter()

    runtime = end - start
    runtime_init = end_spec - start_spec
    runtime_Wirt = end_Wirt - start_Wirt

    if (plot_x == True):
        plot2D(x, to_cpu(x_r), dx_slm)

    return to_cpu(x_r), [runtime, runtime_init, runtime_Wirt]

def plot2D(X_true, X_rec, dx):
    """Generate 2D plot of the true object and reconstructed object
       with axes in physical units."""

    # [FIX 2] angle(<X_rec, X_true>) is the phase X_rec must be rotated BY
    # to match X_true, so multiply by exp(+i theta)
    theta = np.angle(np.vdot(X_rec.ravel(), X_true.ravel()))
    X_rec = X_rec * np.exp(1j * theta)
    X_rec = X_rec * (np.max(np.abs(X_true)) / (np.max(np.abs(X_rec)) + 1e-12))

    amp_true = np.abs(X_true)
    amp_rec = np.abs(X_rec)

    phase_true = np.angle(X_true)
    phase_rec = np.angle(X_rec)

    #amp_vmin = min(amp_true.min(), amp_rec.min())
    amp_vmin = 0
    amp_vmax = max(amp_true.max(), amp_rec.max())

    n = X_true.shape[0]
    L = n * dx

    # Plot in mm
    extent = [-L/2 * 1e3, L/2 * 1e3, -L/2 * 1e3, L/2 * 1e3]

    fig, ax = plt.subplots(2, 2, figsize=(6, 6), constrained_layout=True)

    im1 = ax[0,0].imshow(amp_true, cmap="viridis", vmin=amp_vmin, vmax=amp_vmax, extent=extent, origin="lower")
    ax[0,0].set_title("True Amplitude")

    im2 = ax[0,1].imshow(amp_rec, cmap="viridis", vmin=amp_vmin, vmax=amp_vmax, extent=extent, origin="lower")
    ax[0,1].set_title("Recovered Amplitude")

    im3 = ax[1,0].imshow(phase_true, cmap="twilight", vmin=-np.pi, vmax=np.pi, extent=extent, origin="lower")
    ax[1,0].set_title("True Phase")

    im4 = ax[1,1].imshow(phase_rec, cmap="twilight", vmin=-np.pi, vmax=np.pi, extent=extent, origin="lower")
    ax[1,1].set_title("Recovered Phase")

    for a in ax.ravel():
        a.set_xlabel("x [mm]")
        a.set_ylabel("y [mm]")

    fig.colorbar(im1, ax=ax[0,:], shrink=0.8, label="Amplitude")
    fig.colorbar(im3, ax=ax[1,:], shrink=0.8, label="Phase [rad]")

    plt.show()

In [ ]:
#Real setup, simulated as a scale model: pixel pitches, wavelength and fill factor stay real;
#beam size, camera pixel count and distance are divided by SCALE. This keeps the speckle size on
#the camera, the fraction of light reaching it and the measurements-per-unknown ratio of the real
#setup. Shift, distance, noise, zeroth order, fill factor and aberrations then carry over directly;
#the sweep cells convert rotation, tilt and wavelength errors between real and simulated values.
SCALE = 12                      # scale-model factor; on a GPU you can afford smaller values (8, 6)
                                #   for closer-to-real models: the grid grows roughly as 1/SCALE
z_real = 0.15                   # real SLM-camera distance [m] (0.12-0.18 m suits these parts)
beam_diameter_real = 6e-3       # 1/e^2 intensity diameter of the laser [m]
n_cam_real = 2304               # camera pixels per side

#Resolution and masks
dx_slm = 12.5e-6
n_slm = 2 * int(np.ceil(0.8 * beam_diameter_real / SCALE / dx_slm))   # SLM region 1.6x the beam
L = 16

dx_cam = 6.5e-6
n_cam = int(round(n_cam_real / SCALE))

#Random seed (masks, noise and spectral initialisation are reproducible)
seed = 0

#Iteration and convergence parameters
iter_WF = 1000
iter_SI = 500
tol_WF = 1e-4
tol_SI = 1e-3
tau0 = 20
mu_max = 0.2        # normalised step size (dimensionless, see Wirtinger_iteration)

#Laser
wavelength = 631.8e-9
sigma = beam_diameter_real / SCALE / (2 * np.sqrt(2))   # amplitude sigma of the scaled beam

#Propagation
z = z_real / SCALE

#True system (generates the data). Change these to model errors (wavelength, z, shift, rotation)
wavelength_true = wavelength
z_true = z
shift_x_true = 0.0
shift_y_true = 0.0
rotation_true = 0.0
tilt_x_true = 0.0
tilt_y_true = 0.0

#SLM zeroth order: fraction of the power the SLM leaves unmodulated, and its phase [rad]
#(0 = ideal SLM; real LCoS SLMs are typically a few percent)
zero_order_power = 0.0         # assumed by the reconstruction
zero_order_phase = 0.0
zero_order_power_true = 0.0    # used to generate the data
zero_order_phase_true = 0.0

#Detector model: field computed on a grid `upsample` times finer than the SLM pitch,
#each camera pixel averages the intensity over pixel_subsamples x pixel_subsamples points
upsample = 2
pixel_subsamples = 2

#Numerical precision: keep "single" on the GPU (a T4 runs double precision ~30x slower)
precision = "single"

#SLM fill factor (fraction of each pixel's area that is active; LCoS SLMs are typically 0.9-0.95)
fill_factor = 0.968        # assumed by the reconstruction
fill_factor_true = 0.968   # used to generate the data

#Propagation grid: zero-padded around the SLM, sized for the larger of the true/ideal systems
n_prop = choose_prop_grid(n_slm, dx_slm, n_cam, dx_cam, max(wavelength, wavelength_true), max(z, z_true),
                          shift_max=max(abs(shift_x_true), abs(shift_y_true)), rotation_max_deg=abs(rotation_true),margin=1.0)
print(f"SLM {n_slm*dx_slm*1e3:.2f} mm, camera {n_cam*dx_cam*1e3:.2f} mm, "
      f"propagation grid {n_prop} x {n_prop} ({n_prop*dx_slm*1e3:.2f} mm)")

prop = Propagator(n_slm, n_prop, dx_slm, wavelength_true, z_true, fill_factor=fill_factor_true, upsample=upsample, precision=precision)
prop_ideal = Propagator(n_slm, n_prop, dx_slm, wavelength, z, fill_factor=fill_factor, upsample=upsample, precision=precision)
sampler = SamplingOperator(n_prop=prop.n_out, dx_prop=prop.dx_out, n_det=n_cam, dx_det=dx_cam, shift_x=shift_x_true, shift_y=shift_y_true, rotation_deg=rotation_true, origin=prop.origin_out, tilt_x_deg=tilt_x_true, tilt_y_deg=tilt_y_true, subsamples=pixel_subsamples, precision=precision)
sampler_ideal = SamplingOperator(n_prop=prop_ideal.n_out, dx_prop=prop_ideal.dx_out, n_det=n_cam, dx_det=dx_cam, shift_x=0.0, shift_y=0.0, rotation_deg=0, origin=prop_ideal.origin_out, subsamples=pixel_subsamples, precision=precision)

#Camera noise (values from the camera datasheet). use_noise = False gives noise-free data
use_noise = True
camera_noise = dict(
    full_well = 20000,   # max electrons per pixel [e-]
    peak_fill = 0.8,     # brightest pixel as a fraction of full well (sets exposure)
    read_noise = 3.0,    # read noise [e- rms]
    dark_e = 0.0,        # dark electrons per pixel per frame [e-]
    bit_depth = 12,      # ADC bits
    offset_adu = 100,    # black level [ADU]
)

In [ ]:
x = define_sample(n_slm, sigma, dx_slm,
                  aberrations={"defocus": 5, "astig_0": 2, "coma_x": 1})

In [ ]:
y, D = generate_measurement(x, prop, sampler, camera_noise if use_noise else None, L, n_slm, np.random.default_rng(seed),
                            zero_order_power=zero_order_power_true, zero_order_phase=zero_order_phase_true)

In [ ]:
for j in range(1):
    runtime = 0
    runtime_spec = 0
    runtime_Wirt = 0
    accuracy = 0
    runs = 1
    norm_x = np.sqrt(np.real(np.vdot(x, x)))

    for i in range(runs):
        rng_init = np.random.default_rng(seed + 1 + i)
        # [ZERO ORDER] reconstruct with the masks the model believes the SLM produced
        D_model = effective_masks(D, zero_order_power, zero_order_phase)
        X = reconstruct_x(y, D_model, prop_ideal, sampler_ideal, iter_WF, iter_SI, tol_WF, tol_SI, tau0, mu_max, True, sigma, dx_slm, rng_init)
        runtime += X[1][0]
        runtime_spec += X[1][1]
        runtime_Wirt += X[1][2]
        accuracy += np.abs(np.vdot(X[0], x)) / (np.sqrt(np.real(np.vdot(X[0], X[0]))) * norm_x)

    print(f"Total Runtime   (s): {runtime}")
    print(f"Avg. Runtime    (s): {runtime / runs}")
    print(f"Avg. Runtime SI (s): {runtime_spec / runs}")
    print(f"Avg. Runtime W  (s): {runtime_Wirt / runs}")
    print(f"Accuracy        (%): {accuracy / runs}")

In [ ]:
from matplotlib.patches import Polygon

def visualize_sampling_2d(
    n_prop,
    dx_prop,
    n_det,
    dx_det,
    shift_x=0,
    shift_y=0,
    rotation_deg=0,
    step=16,
    origin=None,
    n_slm=None,
    tilt_x_deg=0,
    tilt_y_deg=0,
):
    """
    Visualize detector sample locations relative
    to the propagation grid.

    Uses the same convention as SamplingOperator: the detector pixel at
    local (Xd, Yd) sits at R(rotation) (Xd, Yd) + shift.
    """

    if origin is None:
        origin = n_prop / 2

    # Propagation grid boundary (the optical axis is at grid index `origin`)
    x_min_prop = -origin * dx_prop
    width_prop = n_prop * dx_prop

    # Detector coordinates
    x_det = (np.arange(n_det) - n_det/2) * dx_det
    y_det = (np.arange(n_det) - n_det/2) * dx_det

    Xd, Yd = np.meshgrid(
        x_det,
        y_det,
        indexing="ij"
    )

    theta = np.deg2rad(rotation_deg)

    ct = np.cos(theta)
    st = np.sin(theta)

    tx, ty = np.deg2rad(tilt_x_deg), np.deg2rad(tilt_y_deg)
    X_r = ct * Xd - st * Yd
    Y_r = st * Xd + ct * Yd
    Xd_rot = X_r * np.cos(ty) + Y_r * np.sin(tx) * np.sin(ty) + shift_x
    Yd_rot = Y_r * np.cos(tx) + shift_y

    plt.figure(figsize=(8, 8))

    # Propagation window
    plt.gca().add_patch(
        plt.Rectangle(
            (x_min_prop, x_min_prop),
            width_prop,
            width_prop,
            fill=False,
            linewidth=3,
            color="blue",
            label="Propagation grid"
        )
    )

    # SLM aperture
    if n_slm is not None:
        width_slm = n_slm * dx_prop
        plt.gca().add_patch(
            plt.Rectangle(
                (-width_slm/2, -width_slm/2),
                width_slm,
                width_slm,
                fill=False,
                linewidth=2,
                color="green",
                label="SLM aperture",
                zorder=3
            )
        )

    plt.scatter(
        Xd_rot[::step, ::step],
        Yd_rot[::step, ::step],
        s=8,
        color="red",
        label="Detector pixels"
    )

    plt.axis("equal")
    plt.grid(True)

    plt.xlabel("x [m]")
    plt.ylabel("y [m]")

    plt.legend()

    plt.title(
        f"dx_prop={dx_prop:.2e}, "
        f"dx_det={dx_det:.2e}"
    )

    plt.show()

In [ ]:
visualize_sampling_2d(
    n_prop=n_prop,
    dx_prop=dx_slm,
    n_det=n_cam,
    dx_det=dx_cam,
    shift_x=shift_x_true,
    shift_y=shift_y_true,
    rotation_deg=rotation_true,
    step = 1,
    origin=prop.origin,
    n_slm=n_slm,
    tilt_x_deg=tilt_x_true,
    tilt_y_deg=tilt_y_true
)

## Tolerance sweeps
Run the settings cell first, then any sweep cell on its own. Each sweep cell runs its simulations, plots them and prints the tolerance.

All sweep values and plots are in **real-setup units**. The simulation is a scale model (`SCALE` in the parameters cell); the settings cell converts rotation, tilt and wavelength errors to the scale model automatically. Shift, distance, noise, zeroth order, fill factor and aberrations carry over unchanged (checked at scales 8, 12 and 16).

In [ ]:
# ===================== TOLERANCE SWEEPS: SETTINGS AND HELPERS =====================
# Run this cell first; each sweep below is then a separate cell with its own plot.
# Every point is a full simulation: data from a "true" system with one error,
# reconstruction with the nominal system from the parameters cell.

import os
import multiprocessing as mp

# ---- speed / accuracy settings for the sweeps -------------------------------
QUICK = False                 # True: 1 seed and a looser stopping rule, ~3x faster (for a first look)

SWEEP = dict(
    n_seeds = 2,              # independent masks, noise and start per point (2 = see the spread)
    upsample = 2,             # keep 2: the Fourier upsampling is what makes the detector model accurate
    pixel_subsamples = 1,     # 1 instead of 2: ~3x faster, camera-model error 3.7 % instead of 1.5 %
    iter_WF = 1000,           # maximum iterations; runs stop by themselves once converged
    iter_SI = 300,            # spectral initialisation iterations (it usually stops earlier)
    mu_max = 0.2,             # normalised step size (stable for 10x brightness and fill factor 0.6 in tests)
    precision = "single",     # ~1.5x faster; results identical to double at the plotted accuracy
    tol_WF = 3e-4,            # looser than 1e-4: ~25 % fewer iterations, same tolerance results
    n_workers = 1 if GPU else os.cpu_count(),   # [GPU] one process: the GPU does the parallel work
                                                # (CUDA does not survive forking into worker processes)
)
if QUICK:
    SWEEP.update(n_seeds=1, tol_WF=1e-3)
phase_rms_limit = 2 * np.pi / 20   # pass/fail line [rad]: lambda/20 rms (0.314 rad, ~32 nm OPD)
base_aberrations = {"defocus": 5, "astig_0": 2, "coma_x": 1}   # keep equal to the beam defined above

# nominal system = the values the reconstruction assumes (parameters cell)
NOMINAL = dict(wavelength=wavelength, z=z, shift_x=0.0, shift_y=0.0, rotation=0.0,
               tilt_x=0.0, tilt_y=0.0, fill_factor=fill_factor,
               zero_order_power=zero_order_power, zero_order_phase=zero_order_phase)

# ---- real setup <-> scale model ------------------------------------------------
# Shift and distance errors carry over unchanged. A smaller (scaled) camera needs a larger
# rotation or tilt to move its edge pixels by the same amount, and a scaled distance needs a
# larger wavelength change for the same effect. Checked against simulations at SCALE = 8, 12, 16.
def sim_rotation(deg_real):
    return deg_real * SCALE

def sim_tilt(deg_real):
    '''same foreshortening W/2 (1 - cos) at the camera edge'''
    return np.degrees(np.arccos(1 - SCALE * (1 - np.cos(np.radians(deg_real)))))

def sim_wavelength_error(dl_real):
    return dl_real * SCALE

def _build_system(p, n_p):
    pr = Propagator(n_slm, n_p, dx_slm, p["wavelength"], p["z"],
                    fill_factor=p["fill_factor"], upsample=SWEEP["upsample"], precision=SWEEP["precision"])
    sa = SamplingOperator(n_prop=pr.n_out, dx_prop=pr.dx_out, n_det=n_cam, dx_det=dx_cam,
                          shift_x=p["shift_x"], shift_y=p["shift_y"], rotation_deg=p["rotation"],
                          origin=pr.origin_out, tilt_x_deg=p["tilt_x"], tilt_y_deg=p["tilt_y"],
                          subsamples=SWEEP["pixel_subsamples"], precision=SWEEP["precision"])
    return pr, sa

def _aberration_basis(n, dx):
    R = n * dx / 2
    c = (np.arange(n) - n/2) * dx
    X, Y = np.meshgrid(c, c, indexing="ij")
    u, v = X / R, Y / R
    r2 = u**2 + v**2
    return {"piston": np.ones_like(u), "tilt_x": u, "tilt_y": v, "defocus": 2*r2 - 1,
            "astig_0": u**2 - v**2, "astig_45": 2*u*v, "coma_x": (3*r2 - 2)*u,
            "coma_y": (3*r2 - 2)*v, "spherical": 6*r2**2 - 6*r2 + 1}

def _metrics(xr, xt):
    '''relative error, amplitude-weighted phase RMS, and the aberration content of the phase error'''
    xr = to_cpu(xr)
    th = np.angle(np.vdot(xr, xt))
    xa = xr * np.exp(1j * th)
    w = np.abs(xt) ** 2
    dphi = np.angle(xa * np.conj(xt))
    basis = _aberration_basis(xt.shape[0], dx_slm)
    A = np.stack([b.ravel() for b in basis.values()], 1)
    sw = np.sqrt(w.ravel())
    coef = np.linalg.lstsq(A * sw[:, None], dphi.ravel() * sw, rcond=None)[0]
    out = dict(rel_err=np.linalg.norm(xa - xt) / np.linalg.norm(xt),
               phase_rms=np.sqrt(np.sum(w * dphi**2) / np.sum(w)))
    out.update({"bias_" + k: c for k, c in zip(basis, coef) if k != "piston"})
    return out

def _solve(y, Dm, pr, sa, rng):
    '''same algorithm as reconstruct_x (accelerated Wirtinger Flow), also returns the iterations used'''
    x0 = spectral_init(y, Dm, pr, sa, SWEEP["iter_SI"], tol_SI, rng)
    return Wirtinger_iteration(x0, y, Dm, pr, sa, SWEEP["iter_WF"], SWEEP["tol_WF"], tau0, SWEEP["mu_max"],
                               return_iters=True)

def run_case(seed, true=None, model=None, aberrations=None, noise="default"):
    '''One simulation. true / model: dicts overriding NOMINAL for the data / the reconstruction.'''
    pt = {**NOMINAL, **(true or {})}
    pm = {**NOMINAL, **(model or {})}
    n_p = choose_prop_grid(n_slm, dx_slm, n_cam, dx_cam, max(pt["wavelength"], pm["wavelength"]),
                           max(pt["z"], pm["z"]),
                           shift_max=max(abs(pt["shift_x"]), abs(pt["shift_y"]), abs(pm["shift_x"]), abs(pm["shift_y"])),
                           rotation_max_deg=max(abs(pt["rotation"]), abs(pm["rotation"])), margin=1.0)
    pr_t, sa_t = _build_system(pt, n_p)
    pr_m, sa_m = _build_system(pm, n_p)

    xt = define_sample(n_slm, sigma, dx_slm, aberrations=base_aberrations if aberrations is None else aberrations)
    nz = (camera_noise if use_noise else None) if noise == "default" else noise

    # same steps as generate_measurement (without its timing print)
    rng = np.random.default_rng(seed)
    D_prog = define_masks(L, n_slm, rng)
    y_meas = y_i(effective_masks(D_prog, pt["zero_order_power"], pt["zero_order_phase"]), xt, pr_t, sa_t)
    if nz is not None:
        y_meas = add_camera_noise(y_meas, rng, **nz)

    D_mod = effective_masks(D_prog, pm["zero_order_power"], pm["zero_order_phase"])
    xr, iters = _solve(y_meas, D_mod, pr_m, sa_m, np.random.default_rng(seed + 1000))
    m = _metrics(xr, xt)
    m["iters"] = iters
    return m

_SWEEP_CASES = []

def _sweep_job(args):
    i, s = args
    try:
        pyfftw.config.NUM_THREADS = 1      # one FFT thread per process when running in parallel
    except NameError:
        pass
    return run_case(seed + s, **_SWEEP_CASES[i][2])

def run_sweep(cases):
    '''cases: list of (curve label, x value, run_case keyword arguments).
       Returns {label: {"x": [...], metric: array (points x seeds)}} sorted by x.'''
    global _SWEEP_CASES
    _SWEEP_CASES = cases
    jobs = [(i, s) for i in range(len(cases)) for s in range(SWEEP["n_seeds"])]
    t0 = time.perf_counter()
    results = None
    if SWEEP["n_workers"] > 1:
        try:
            with mp.get_context("fork").Pool(SWEEP["n_workers"]) as pool:
                results = pool.map(_sweep_job, jobs)
        except (ValueError, OSError, RuntimeError) as err:
            print(f"parallel run unavailable ({err}); running sequentially")
    if results is None:
        results = [_sweep_job(j) for j in jobs]
    print(f"  {len(jobs)} simulations in {time.perf_counter() - t0:.0f} s")

    out = {}
    for (i, s), r in zip(jobs, results):
        label, xval, _ = cases[i]
        cur = out.setdefault(label, {})
        cur.setdefault(xval, []).append(r)
    curves = {}
    for label, pts in out.items():
        xs = sorted(pts)
        curves[label] = {"x": np.array(xs)}
        for key in pts[xs[0]][0]:
            curves[label][key] = np.array([[r[key] for r in pts[xv]] for xv in xs])
    return curves

def tolerance(xs, ys, limit=None):
    '''smallest x at which the mean curve crosses the limit (linear interpolation)'''
    limit = phase_rms_limit if limit is None else limit
    ys = np.nan_to_num(np.asarray(ys, dtype=float), nan=np.inf)   # a failed (NaN) run counts as failing
    for i in range(1, len(xs)):
        if ys[i] > limit >= ys[i - 1]:
            return xs[i-1] + (limit - ys[i-1]) * (xs[i] - xs[i-1]) / (ys[i] - ys[i-1])
        if ys[i] > limit:
            return xs[i]
    return np.inf

def plot_curves(ax, curves, xlabel, title, xscale=1.0, logx=False):
    for label, c in curves.items():
        mean = c["phase_rms"].mean(1)
        err = [mean - c["phase_rms"].min(1), c["phase_rms"].max(1) - mean]
        ax.errorbar(c["x"] * xscale, mean, yerr=err, marker="o", ms=4, capsize=3, label=label)
    ax.axhline(phase_rms_limit, color="k", ls="--", lw=1, label=f"limit λ/20 = {phase_rms_limit:.3f} rad")
    if "baseline" in globals():
        ax.axhline(BASELINE["phase_rms"].mean(), color="0.35", ls="-.", lw=1.4, label="perfect system")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("phase RMS error [rad]")
    ax.set_title(title)
    if logx:
        ax.set_xscale("log")
    ax.minorticks_on()
    ax.grid(which="major", alpha=0.35)
    ax.grid(which="minor", alpha=0.15, lw=0.5)
    ax.legend(fontsize=8)

def plot_sweep(curves, xlabel, title, xscale=1.0, logx=False):
    '''one figure per sweep'''
    fig, ax = plt.subplots(figsize=(7, 4.5), constrained_layout=True)
    plot_curves(ax, curves, xlabel, title, xscale=xscale, logx=logx)
    return fig, ax

def report(curves, unit, xscale=1.0, extra=None):
    for label, c in curves.items():
        tol = tolerance(c["x"] * xscale, c["phase_rms"].mean(1))
        tol_txt = f"> {c['x'].max() * xscale:g} {unit} (never fails)" if np.isinf(tol) else f"{tol:.3g} {unit}"
        line = f"    {label:28s} tolerance: {tol_txt}"
        if extra:
            line += "   " + extra(c)
        print(line)
        convergence_check(label, c)

def convergence_check(label, c):
    '''Runs that hit the iteration limit matter only if they are below the limit:
       failing runs (far above it) stay failing with more iterations.'''
    if np.any(~np.isfinite(c["phase_rms"])):
        print(f"    WARNING: some '{label}' runs failed (NaN); they are counted as failing")
    capped = c["iters"] >= SWEEP["iter_WF"]
    if np.any(capped & (c["phase_rms"] < phase_rms_limit)):
        print(f"    WARNING: some passing '{label}' runs hit the {SWEEP['iter_WF']}-iteration limit; "
              f"increase SWEEP['iter_WF'] to check that tolerance")

def with_baseline(curves):
    '''add the perfect-system point at x = 0 to every curve'''
    for c in curves.values():
        for key in c:
            if key == "x":
                c[key] = np.r_[0.0, c[key]]
            else:
                c[key] = np.vstack([BASELINE[key][None, :], c[key]])
    return curves

# ---- perfect system (shared x = 0 point) ------------------------------------
t_start = time.perf_counter()
print("Perfect system (baseline)")
baseline = run_sweep([("baseline", 0.0, {})])["baseline"]
BASELINE = {k: v[0] for k, v in baseline.items() if k != "x"}
print(f"    phase RMS {BASELINE['phase_rms'].mean():.3f} rad, rel err {BASELINE['rel_err'].mean():.3f}")
print(f"    {(time.perf_counter() - t_start):.0f} s for {SWEEP['n_seeds']} simulation(s); "
      f"each sweep cell below runs 8-24 points")

### 1) Lateral camera shift

In [ ]:
# ---- 1) Lateral camera shift (real units; carries over unchanged) -------------------
vals = [1e-6, 2e-6, 3e-6, 4e-6, 5e-6, 6e-6]
sweep_shift = with_baseline(run_sweep(
    [("shift x", v, dict(true=dict(shift_x=v))) for v in vals] +
    [("shift y", v, dict(true=dict(shift_y=v))) for v in vals]))

plot_sweep(sweep_shift, "camera shift [µm]", "1) Lateral camera shift", xscale=1e6)
plt.show()
report(sweep_shift, "µm", 1e6)

### 2) SLM–camera distance

In [ ]:
# ---- 2) SLM-camera distance (real units; carries over unchanged) --------------------
vals = [0.05e-3 * k for k in range(1, 13)]          # 0.05 ... 0.6 mm
sweep_z = with_baseline(run_sweep(
    [("camera farther (+dz)", v, dict(true=dict(z=z + v))) for v in vals] +
    [("camera closer (-dz)", v, dict(true=dict(z=z - v))) for v in vals]))

plot_sweep(sweep_z, "|distance error| [mm]", "2) SLM-camera distance", xscale=1e3)
plt.show()
report(sweep_z, "mm", 1e3)

### 5) In-plane rotation

In [ ]:
# ---- 5) In-plane rotation (real degrees, converted to the scale model) -------------
vals = [0.025, 0.05, 0.1, 0.15, 0.2, 0.3]
sweep_rot = with_baseline(run_sweep([("rotation", v, dict(true=dict(rotation=sim_rotation(v)))) for v in vals]))

plot_sweep(sweep_rot, "rotation [deg]", "5) In-plane rotation")
plt.show()
report(sweep_rot, "deg")

### 6) Camera tilt

In [ ]:
# ---- 6) Camera tilt (real degrees, converted to the scale model) -------------------
vals = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0]
sweep_tilt = with_baseline(run_sweep(
    [("tilt about x", v, dict(true=dict(tilt_x=sim_tilt(v)))) for v in vals] +
    [("tilt about y", v, dict(true=dict(tilt_y=sim_tilt(v)))) for v in vals] +
    [("tilt about x and y", v, dict(true=dict(tilt_x=sim_tilt(v), tilt_y=sim_tilt(v)))) for v in vals]))

plot_sweep(sweep_tilt, "tilt [deg]", "6) Camera tilt")
plt.show()
report(sweep_tilt, "deg", extra=lambda c: "astigmatism bias at max tilt: "
       f"{c['bias_astig_0'][-1].mean():+.3f} rad (0 deg), {c['bias_astig_45'][-1].mean():+.3f} rad (45 deg)")

### 3) Zeroth-order leak

In [ ]:
# ---- 3) Zeroth-order leak -----------------------------------------------------
vals = [0.01, 0.02, 0.05, 0.10]
sweep_zo = with_baseline(run_sweep(
    [("model ignores it", v, dict(true=dict(zero_order_power=v), model=dict(zero_order_power=0.0))) for v in vals] +
    [("model knows it", v, dict(true=dict(zero_order_power=v), model=dict(zero_order_power=v))) for v in vals] +
    [("known power, phase off by 90°", v, dict(true=dict(zero_order_power=v, zero_order_phase=np.pi/2),
                                                model=dict(zero_order_power=v, zero_order_phase=0.0))) for v in vals]))

plot_sweep(sweep_zo, "unmodulated power [%]", "3) Zeroth-order leak", xscale=100)
plt.show()
report(sweep_zo, "% power", 100)

### 4) Noise / exposure

In [ ]:
# ---- 4) Noise / exposure --------------------------------------------------------
fills = [0.01, 0.03, 0.1, 0.3, 0.9]
noise_cases = []
for fw in [2000, 20000]:
    for pf in fills:
        nz = {**camera_noise, "full_well": fw, "peak_fill": pf}
        noise_cases.append((f"full well {fw} e-", pf * fw, dict(noise=nz)))   # x = electrons in the brightest pixel
sweep_noise = run_sweep(noise_cases)

plot_sweep(sweep_noise, "electrons in the brightest pixel", "4) Noise / exposure", logx=True)
plt.show()
print("    phase RMS by electrons in the brightest pixel")
for label, c in sweep_noise.items():
    print(f"      {label:18s} " + ", ".join(f"{x:g} e-: {p:.3f}" for x, p in zip(c["x"], c["phase_rms"].mean(1))))
    convergence_check(label, c)

### 7) Fill-factor mismatch

In [ ]:
# ---- 7) Fill-factor mismatch ------------------------------------------------------
vals = [0.80, 0.85, 0.90, 0.95, 1.00]
sweep_ff = run_sweep([(f"model assumes {fill_factor:g}", v, dict(true=dict(fill_factor=v))) for v in vals])

plot_sweep(sweep_ff, "true SLM fill factor", "7) Fill-factor mismatch")
plt.show()
for label, c in sweep_ff.items():
    print(f"    {label}: " + ", ".join(f"{x:.2f}: {p:.3f} rad" for x, p in zip(c["x"], c["phase_rms"].mean(1))))
    convergence_check(label, c)

### 8) Beam aberrations

In [ ]:
# ---- 8) Beam aberrations (one term changed at a time, others as in base_aberrations) ----
beam_cases = []
for term, values in [("defocus", [0, 15, 30]), ("astig_0", [0, 5, 10]), ("coma_x", [0, 5, 10])]:
    for v in values:
        beam_cases.append((term, float(v), dict(aberrations={**base_aberrations, term: v})))
sweep_beam = run_sweep(beam_cases)

def max_phase_step(ab):
    '''largest phase change between neighbouring SLM pixels [rad] (must stay well below pi)'''
    ph = np.angle(define_sample(n_slm, sigma, dx_slm, aberrations=ab))
    d = [np.angle(np.exp(1j * np.diff(ph, axis=a))) for a in (0, 1)]
    return max(np.abs(d[0]).max(), np.abs(d[1]).max())

plot_sweep(sweep_beam, "aberration coefficient [rad]", "8) Beam aberrations (one term varied)")
plt.show()
print("    phase RMS and largest phase step per SLM pixel (results are not meaningful above ~pi/2 per pixel)")
for term, c in sweep_beam.items():
    for x, p in zip(c["x"], c["phase_rms"].mean(1)):
        step = max_phase_step({**base_aberrations, term: x})
        flag = "  <-- over pi/2 per pixel" if step > np.pi / 2 else ""
        print(f"      {term:8s} = {x:4g} rad: phase RMS {p:.3f} rad, max step {step:.2f} rad/pixel{flag}")
    convergence_check(term, c)

### 9) Wavelength

In [ ]:
# ---- 9) Wavelength ---------------------------------------------------------------
vals = [-0.5e-9, -0.2e-9, 0.2e-9, 0.5e-9]
sweep_wl = run_sweep([("wavelength error", v, dict(true=dict(wavelength=wavelength + sim_wavelength_error(v))))
                      for v in vals])   # real Δλ, converted to the scale model
c = sweep_wl["wavelength error"]                  # insert the perfect system at 0 (middle of the curve)
i0 = np.searchsorted(c["x"], 0.0)
for key in list(c):
    c[key] = np.insert(c[key], i0, 0.0, axis=0) if key == "x" else np.insert(c[key], i0, BASELINE[key], axis=0)

fig, ax = plot_sweep(sweep_wl, "wavelength error [nm]", "9) Wavelength", xscale=1e9)
top = ax.secondary_xaxis("top", functions=(lambda d: d * 1e-9 / wavelength * z_real * 1e3,
                                           lambda dz: dz * 1e-3 / z_real * wavelength * 1e9))
top.set_xlabel("equivalent distance error z·Δλ/λ [mm]")
plt.show()
print("    " + ", ".join(f"{x*1e9:+.2f} nm: {p:.3f} rad" for x, p in zip(c["x"], c["phase_rms"].mean(1))))
convergence_check("wavelength error", c)